# Demo - datasets - training datasets for PARNET

Requires: execution of the notebook `prepare_datasets.py.ipynb`

## Set-up

### Imports

In [1]:
import pylbsr.notebooks
import pylbsr.misc
from dotmap import DotMap
import gzip
import pandas as pd
import numpy as np
import torch
import torch.utils.data
import parnet
import pylbsr.torch_utils
import yaml
import parnet_additional_utils
import parnet_analyses_libs
import json
from functools import partial
from typing import TypedDict, Protocol
from tqdm import tqdm

/home/l10n/projects/parnet-project/parnet--demo--train-models/.pixi/envs/parnet-dev-cu11/lib/python3.10/site-packages/gin/config.py:615: FutureWarning: `NLLLoss2d` has been deprecated. Please use `NLLLoss` instead as a drop-in replacement and see https://pytorch.org/docs/main/nn.html#torch.nn.NLLLoss for more details.
  decorated_class = decorating_meta(cls.__name__, (cls,), overrides)
Seed set to 42
/home/l10n/projects/parnet-project/parnet--demo--train-models/.pixi/envs/parnet-dev-cu11/lib/python3.10/site-packages/parnet/constants.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the function

### Parameters

In [2]:
_notebook_name = "prepare_datasets.py.ipynb"
_notebook_path = f"notebooks/demo--train-spliceosome-hepg2/{_notebook_name}"

### Initialization

In [3]:
pylbsr.misc.set_seed(42)

Seed set to 42


In [4]:
print(torch.cuda.device_count())

# List GPUs
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


2
0 NVIDIA GeForce RTX 5090
1 NVIDIA RTX 4000 SFF Ada Generation


/home/l10n/projects/parnet-project/parnet--demo--train-models/.pixi/envs/parnet-dev-cu11/lib/python3.10/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_35 sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_89 compute_89.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [5]:
# Set cuda device to "1"
torch.cuda.set_device(1)

In [6]:
logger = pylbsr.misc.init_logger(_notebook_name)

PROJECT_DIR = pylbsr.notebooks.find_project_root_from_notebook_path(_notebook_path)

logger.info(f"Project directory: {PROJECT_DIR}")

[14:35:08] INFO - Project directory: /home/l10n/projects/parnet-project/parnet--demo--train-models


### Filepaths

In [ ]:
from pathlib import Path

_fp_cfg = yaml.safe_load((PROJECT_DIR / "config" / "filepaths.yaml").read_text())


def _res(p):
    p = Path(p)
    return p if p.is_absolute() else PROJECT_DIR / p


FILEPATHS = DotMap()

FILEPATHS["parnet_data_v2_full"] = {
    "metadata": _res(_fp_cfg["parnet_encore_eclip"]["metadata"]),
    "data": _res(_fp_cfg["nas"]["parnet_encore_eclip_data_2000nt"]),
}

FILEPATHS["parnet_data_v1_full"] = {
    # NAS-only (600 nt HuggingFace dataset format; not used in current demo)
    "folder": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_training_data/600nt_windows/encode.filtered.hfds/",
    "folder": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_training_data/600nt_windows/encode.filtered.hfds/dataset_dict.json",
    #
    "train_folder": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_training_data/600nt_windows/encode.filtered.hfds/train/",
    "train_ufmt_arrow": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_training_data/600nt_windows/encode.filtered.hfds/train/data-{IDX_OF_TOTAL}.arrow",
    "train_state_json": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_training_data/600nt_windows/encode.filtered.hfds/train/state.json",
    "train_info_json": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_training_data/600nt_windows/encode.filtered.hfds/train/dataset_info.json",
    #
    "test_folder": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_testing_data/600nt_windows/encode.filtered.hfds/test/",
    "test_ufmt_arrow": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_testing_data/600nt_windows/encode.filtered.hfds/test/data-{IDX_OF_TOTAL}.arrow",
    "test_state_json": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_testing_data/600nt_windows/encode.filtered.hfds/test/state.json",
    "test_info_json": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_testing_data/600nt_windows/encode.filtered.hfds/test/dataset_info.json",
    #
    "validation_folder": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_validationing_data/600nt_windows/encode.filtered.hfds/validation/",
    "validation_ufmt_arrow": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_validationing_data/600nt_windows/encode.filtered.hfds/validation/data-{IDX_OF_TOTAL}.arrow",
    "validation_state_json": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_validationing_data/600nt_windows/encode.filtered.hfds/validation/state.json",
    "validation_info_json": "/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_validationing_data/600nt_windows/encode.filtered.hfds/validation/dataset_info.json",
}

In [ ]:
FILEPATHS["parnet_models"] = {}
FILEPATHS["parnet_models"]["metadata_config_models"] = (
    PROJECT_DIR / "config" / "parnet_models_metadata.yaml"
)
FILEPATHS["parnet_models"]["metadata_data_splits"] = _res(
    _fp_cfg["metadata"]["train_val_test_splits"]
)
FILEPATHS["parnet_models"]["parnet.7m-0.0"] = _res(_fp_cfg["models"]["parnet.7m-0.0"])
FILEPATHS["parnet_models"]["parnet.21m-0.0"] = _res(_fp_cfg["models"]["parnet.21m-0.0"])

In [ ]:
FILEPATHS["parnet_rbp_metadata"] = {}
FILEPATHS["parnet_rbp_metadata"]["full_rbp_set"] = _res(_fp_cfg["metadata"]["full_rbp_set"])
FILEPATHS["parnet_rbp_metadata"]["metadata_functions"] = _res(_fp_cfg["metadata"]["rbp_functions"])

## Misc load

## Demo - load from TFDS

## Demo - load from .pt

## Process - Generate new dataset with filtered RBPs

### RBPs of interest

In [10]:
rbps_ct_df = pd.read_csv(FILEPATHS["parnet_rbp_metadata"]["full_rbp_set"], sep="\t")
display(rbps_ct_df.head(3))
display(rbps_ct_df.groupby("ct").size())

,rbp_ct,rbp,ct
0,AARS_K562,AARS,K562
1,AATF_K562,AATF,K562
2,ABCF1_K562,ABCF1,K562


ct
HepG2    103
K562     120
dtype: int64

In [11]:
rbps_metadata_function = pd.read_csv(FILEPATHS["parnet_rbp_metadata"]["metadata_functions"])
display(rbps_metadata_function.head(3))

,name,geneID,Essential Genes,Splicing regulation,Spliceosome,RNA modification,3' end processing,rRNA processing,Ribosome & basic translation,RNA stability & decay,...,RNA export,Translation regulation,tRNA regulation,mitochondrial RNA regulation,Viral RNA regulation,snoRNA / snRNA / telomerase,P-body / stress granules,Exon Junction Complex,Novel RBP,Other
0,A1CF,ENSG00000148584,0,0,0,1,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
1,AARS,ENSG00000090861,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,AATF,ENSG00000108270,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [12]:
subset_rbp_from_spliceosome = rbps_metadata_function[rbps_metadata_function["Spliceosome"] == 1]
display(subset_rbp_from_spliceosome.head(3))
print(f"Number of RBPs in spliceosome: {len(subset_rbp_from_spliceosome)}")

# Now check how many of these are in HepG2 and K562.
rbps_in_spliceosome = set(subset_rbp_from_spliceosome["name"])
rbps_ct_in_spliceosome = rbps_ct_df[rbps_ct_df["rbp"].isin(rbps_in_spliceosome)]
display(rbps_ct_in_spliceosome.groupby("ct").size())


# Now show the overlapping sets between the two cell types.
# For the subset in spliceosome.
rbps_k562_in_spliceosome = set(rbps_ct_in_spliceosome[rbps_ct_in_spliceosome["ct"] == "K562"]["rbp"])
rbps_hepg2_in_spliceosome = set(
    rbps_ct_in_spliceosome[rbps_ct_in_spliceosome["ct"] == "HepG2"]["rbp"]
)
print(f"RBPs in spliceosome and K562: {len(rbps_k562_in_spliceosome)}")
print(f"RBPs in spliceosome and HepG2: {len(rbps_hepg2_in_spliceosome)}")
print(
    f"RBPs in spliceosome and both K562 and HepG2: {len(rbps_k562_in_spliceosome & rbps_hepg2_in_spliceosome)}"
)

print(f"RBPs in spliceosome and only K562: {(rbps_k562_in_spliceosome - rbps_hepg2_in_spliceosome)}")
print(f"RBPs in spliceosome and only HepG2: {(rbps_hepg2_in_spliceosome - rbps_k562_in_spliceosome)}")
print(
    f"RBPs in spliceosome and both K562 and HepG2: {(rbps_k562_in_spliceosome & rbps_hepg2_in_spliceosome)}"
)

,name,geneID,Essential Genes,Splicing regulation,Spliceosome,RNA modification,3' end processing,rRNA processing,Ribosome & basic translation,RNA stability & decay,...,RNA export,Translation regulation,tRNA regulation,mitochondrial RNA regulation,Viral RNA regulation,snoRNA / snRNA / telomerase,P-body / stress granules,Exon Junction Complex,Novel RBP,Other
16,AQR,ENSG00000021776,1,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
26,BUD13,ENSG00000137656,1,1,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
29,CCAR1,ENSG00000060339,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


Number of RBPs in spliceosome: 35


ct
HepG2    14
K562     13
dtype: int64

RBPs in spliceosome and K562: 13
RBPs in spliceosome and HepG2: 14
RBPs in spliceosome and both K562 and HepG2: 9
RBPs in spliceosome and only K562: {'DDX42', 'GEMIN5', 'SF3B1', 'GPKOW'}
RBPs in spliceosome and only HepG2: {'SF3A3', 'PRPF4', 'CDC40', 'RBM5', 'SFPQ'}
RBPs in spliceosome and both K562 and HepG2: {'RBM22', 'U2AF2', 'BUD13', 'EFTUD2', 'SMNDC1', 'U2AF1', 'AQR', 'PRPF8', 'SF3B4'}


In [13]:
subset_rbp_spliceosome_both = rbps_k562_in_spliceosome & rbps_hepg2_in_spliceosome
print(f"RBPs in spliceosome and both K562 and HepG2: {len(subset_rbp_spliceosome_both)}")
print(f"RBPs in spliceosome and both K562 and HepG2: {subset_rbp_spliceosome_both}")

RBPs in spliceosome and both K562 and HepG2: 9
RBPs in spliceosome and both K562 and HepG2: {'RBM22', 'U2AF2', 'BUD13', 'EFTUD2', 'SMNDC1', 'U2AF1', 'AQR', 'PRPF8', 'SF3B4'}


### Definitions

In [ ]:
class SparseTensorDict(TypedDict):
    indices: torch.Tensor  # shape (2, nnz)
    values: torch.Tensor  # shape (nnz,)
    size: torch.Size  # e.g. (N_TRACKS, LENGTH)


class ParnetDataElementInputs(TypedDict):
    sequence: str


class ParnetDataElementOutputs(TypedDict):
    eCLIP: SparseTensorDict
    control: SparseTensorDict


class ParnetDataElement(TypedDict):
    inputs: ParnetDataElementInputs
    outputs: ParnetDataElementOutputs
    meta: dict


def torch_sparse_to_dense(sparse: SparseTensorDict) -> torch.Tensor:
    return torch.sparse_coo_tensor(sparse["indices"], sparse["values"], sparse["size"]).to_dense()


def torch_dense_to_sparse(dense: torch.Tensor) -> SparseTensorDict:
    indices = dense.nonzero(as_tuple=False).T
    values = dense[indices[0], indices[1]]
    return {
        "indices": indices,
        "values": values,
        "size": torch.Size(dense.shape),
    }

In [15]:
class FilterFunction(Protocol):
    def __call__(self, parnet_data_element: "ParnetDataElement", **kwargs) -> bool: ...


def filter_minimum_length(parnet_data_element: "ParnetDataElement", min_length: int) -> bool:
    return len(parnet_data_element["inputs"]["sequence"]) >= min_length


def filter_min_read_count_for_any_track_in_track_indices(
    parnet_data_element: "ParnetDataElement",
    min_read_count: int,
    tasks: list[str],
    track_indices: list[int],
) -> bool:
    """Keep sample if ANY of the selected tracks reaches min_read_count reads in ANY task."""
    for task in tasks:
        dense = torch_sparse_to_dense(parnet_data_element["outputs"][task])
        if dense[track_indices, :].max().item() >= min_read_count:
            return True
    return False

In [ ]:
class FilteredMultiTaskDataset(torch.utils.data.Dataset):
    """Filters a list of ParnetDataElements and extracts a subset of output tracks.

    Adapted from single-RBP FilteredTrackDataset; supports multi-track selection.
    On construction, precomputes which items pass all filter functions (fast iteration).
    """

    def __init__(self, base_list: list, track_indices: list[int], filters: list):
        self.base_list = base_list
        self.track_indices = track_indices
        self.index_map = [i for i, elem in enumerate(base_list) if all(f(elem) for f in filters)]

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx: int) -> "ParnetDataElement":
        elem = self.base_list[self.index_map[idx]]
        outputs = {}
        for task, sparse in elem["outputs"].items():
            dense = torch_sparse_to_dense(sparse)
            outputs[task] = torch_dense_to_sparse(dense[self.track_indices, :])
        return {"inputs": elem["inputs"], "outputs": outputs, "meta": elem["meta"]}


def center_crop_item(
    item: "ParnetDataElement", crop_start: int, crop_end: int
) -> "ParnetDataElement":
    """Crop a data item's sequence and signal tensors to [crop_start:crop_end]."""
    outputs_cropped = {
        task: torch_dense_to_sparse(torch_sparse_to_dense(sparse)[:, crop_start:crop_end])
        for task, sparse in item["outputs"].items()
    }
    return {
        "inputs": {"sequence": item["inputs"]["sequence"][crop_start:crop_end]},
        "outputs": outputs_cropped,
        "meta": item["meta"],
    }

### Process

In [17]:
with open(FILEPATHS["parnet_models"]["metadata_data_splits"], "r") as f:
    data_splits = yaml.safe_load(f)

display(data_splits)

{'test': ['chr3', 'chr8'],
 'validation': ['chr2', 'chr9', 'chr16'],
 'train': ['chr1',
  'chr4',
  'chr5',
  'chr6',
  'chr7',
  'chr10',
  'chr11',
  'chr12',
  'chr13',
  'chr14',
  'chr15',
  'chr17',
  'chr18',
  'chr19',
  'chr20']}

In [ ]:
# === Track selection ===
# Use the 9 spliceosome RBPs shared between HepG2 and K562 (computed above).
selected_hepg2_rbp_cts = sorted(
    [
        f"{rbp}_HepG2"
        for rbp in subset_rbp_spliceosome_both
        if f"{rbp}_HepG2" in rbps_ct_df["rbp_ct"].values
    ]
)
track_indices = [rbps_ct_df["rbp_ct"].tolist().index(rbp_ct) for rbp_ct in selected_hepg2_rbp_cts]
print(f"Selected {len(selected_hepg2_rbp_cts)} HepG2 spliceosome RBPs:")
for rbp_ct, idx in zip(selected_hepg2_rbp_cts, track_indices):
    print(f"  track {idx:3d}  {rbp_ct}")

# === Center-crop: 2000 nt → center 600 nt ===
# The pretrained parnet.7m-0.0 was trained on 600 nt windows.
# We extract the central 600 nt from each 2000 nt tile, discarding the flanking regions.
PARAMS_SEQ_LEN_FULL = 2000
PARAMS_SEQ_LEN_CROP = 600
CROP_START = (PARAMS_SEQ_LEN_FULL - PARAMS_SEQ_LEN_CROP) // 2  # 700
CROP_END = CROP_START + PARAMS_SEQ_LEN_CROP  # 1300
print(f"\nCenter crop: positions [{CROP_START}:{CROP_END}]  ({PARAMS_SEQ_LEN_CROP} nt kept)")

# === Quality filter: minimum 3 reads in any selected track ===
params_min_count_any_track = 3
params_raw_output_keys = ["eCLIP", "control"]

filters = [
    partial(filter_minimum_length, min_length=PARAMS_SEQ_LEN_FULL),
    partial(
        filter_min_read_count_for_any_track_in_track_indices,
        min_read_count=params_min_count_any_track,
        tasks=params_raw_output_keys,
        track_indices=track_indices,
    ),
]

# === Demo subset: use 3 chromosomes per split to keep execution fast ===
filter_chromosomes_to_retain = {
    "train": data_splits["train"][:3],  # e.g. ['chr1', 'chr4', 'chr5']
    "validation": data_splits["validation"][:3],  # e.g. ['chr2', 'chr9', 'chr16']
    "test": data_splits["test"][:3],  # ['chr3', 'chr8'] (only 2 available)
}
print(f"\nChromosome subsets:")
for split, chroms in filter_chromosomes_to_retain.items():
    print(f"  {split}: {chroms}")

Selected 9 HepG2 spliceosome RBPs:
  track   9  AQR_HepG2
  track  13  BUD13_HepG2
  track  41  EFTUD2_HepG2
  track 131  PRPF8_HepG2
  track 144  RBM22_HepG2
  track 159  SF3B4_HepG2
  track 165  SMNDC1_HepG2
  track 193  U2AF1_HepG2
  track 195  U2AF2_HepG2

Center crop: positions [700:1300]  (600 nt kept)

Chromosome subsets:
  train: ['chr1', 'chr4', 'chr5']
  validation: ['chr2', 'chr9', 'chr16']
  test: ['chr3', 'chr8']


In [19]:
# .pt split keys are 'train', 'valid', 'test' (note: 'valid', not 'validation')
_split_key_map = {"train": "train", "validation": "valid", "test": "test"}

output_filepath = PROJECT_DIR / "results" / "spliceosome-hepg2" / "dataset.pt"

# Load the full dataset into memory (~64 GB RAM, ~4 min on this machine)
logger.info("Loading full 2000 nt .pt dataset — this may take a few minutes …")
parnet_data = torch.load(FILEPATHS["parnet_data_v2_full"]["data"])
logger.info(f"Loaded splits: {list(parnet_data.keys())}")

# Process each split with chromosome and quality filters
output_splits = {}
for split_name, chromosomes in filter_chromosomes_to_retain.items():
    pt_key = _split_key_map[split_name]
    base_list = [
        item for item in parnet_data[pt_key] if item["meta"]["name"].split(":")[0] in chromosomes
    ]
    logger.info(f"\n--- {split_name} | chromosomes: {chromosomes} | windows: {len(base_list)} ---")

    ds = FilteredMultiTaskDataset(base_list, track_indices, filters)
    logger.info(f"  After quality filter: {len(ds)} retained")

    output_splits[pt_key] = [
        center_crop_item(ds[i], CROP_START, CROP_END)
        for i in tqdm(range(len(ds)), desc=f"Processing {split_name}")
    ]

del parnet_data  # free ~64 GB RAM

# Save with metadata so the training notebook can reload the RBP order
output_filepath.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        **output_splits,
        "metadata": {
            "rbp_cts": selected_hepg2_rbp_cts,
            "track_indices_in_full_dataset": track_indices,
            "num_tasks": len(selected_hepg2_rbp_cts),
            "crop_start": CROP_START,
            "crop_end": CROP_END,
            "source_data": str(FILEPATHS["parnet_data_v2_full"]["data"]),
        },
    },
    output_filepath,
)
logger.info(f"\nSaved filtered dataset → {output_filepath}")
for k, items in output_splits.items():
    logger.info(f"  {k}: {len(items)} items")

[14:35:08] INFO - Loading full 2000 nt .pt dataset — this may take a few minutes …
/tmp/ipykernel_32285/2279626554.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  parnet

In [20]:
# Quick verification: inspect one saved sample
saved = torch.load(output_filepath)
print("Metadata:", saved["metadata"])
sample = saved["train"][0]
print(
    f"\nSample sequence length : {len(sample['inputs']['sequence'])} nt  (expected {PARAMS_SEQ_LEN_CROP})"
)
for task, sparse in sample["outputs"].items():
    dense = torch_sparse_to_dense(sparse)
    print(
        f"Output '{task}' shape  : {tuple(dense.shape)}  (expected ({len(selected_hepg2_rbp_cts)}, {PARAMS_SEQ_LEN_CROP}))"
    )
del saved

/tmp/ipykernel_32285/3398657320.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved = torch.load(output_filepath)


Metadata: {'rbp_cts': ['AQR_HepG2', 'BUD13_HepG2', 'EFTUD2_HepG2', 'PRPF8_HepG2', 'RBM22_HepG2', 'SF3B4_HepG2', 'SMNDC1_HepG2', 'U2AF1_HepG2', 'U2AF2_HepG2'], 'track_indices_in_full_dataset': [9, 13, 41, 131, 144, 159, 165, 193, 195], 'num_tasks': 9, 'crop_start': 700, 'crop_end': 1300, 'source_data': '/mnt/storage-nas-fast-2/research/projects/hzm/parnet-analyses/MANUAL/parnet_encore_eclip/parnet_preprocessed_training_data/2000nt_windows/gencode.v48.annotation.transcripts.merged.tiles.data.filtered.splits.pt'}

Sample sequence length : 600 nt  (expected 600)
Output 'eCLIP' shape  : (9, 600)  (expected (9, 600))
Output 'control' shape  : (9, 600)  (expected (9, 600))


In [ ]:
# Also export a table tsv file (as the one we used as input) with only the selected RBPs
output_table_rbps = output_filepath.parent / "rbp_cts.tsv"
rbps_ct_df.loc[rbps_ct_df["rbp_ct"].isin(selected_hepg2_rbp_cts)].to_csv(
    output_table_rbps, sep="\t", index=False
)